In [4]:
! uv pip install -U langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community google-genai langchain-google-genai Textloader

Using Python 3.12.2 environment at: e:\projects\llmops_series\.venv
Resolved 91 packages in 2.83s
Prepared 2 packages in 441ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 20ms
 + chardet==5.2.0
 + textloader==0.0.9


In [24]:
import os
from dotenv import load_dotenv

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")


### Data Ingestion

In [7]:
from langchain_community.document_loaders import TextLoader

In [25]:
loader = TextLoader(r"E:\projects\llmops_series\data\Agentic_AI.txt", encoding="utf8")
documents = loader.load()
print(f"Loaded {len(documents)} document(s).")
print(documents[0].page_content[:500])  # print first 500 chars of first document


Loaded 1 document(s).
UNDERSTANDING AGENTIC AI
Definition
Agentic AI refers to systems that act autonomously to pursue objectives in environments by perceiving inputs, planning sequences of actions, and executing those actions. Unlike single-turn or purely reactive models, agentic systems maintain state, set sub-goals, chain multiple steps, and can manage decision-making, tool use, and iterative refinement of outputs.
Core Capabilities
Perception & Input Handling
Ingests multi-modal inputs (text, structured data, fil


In [13]:
import langchain
print(langchain.__version__)


1.0.1


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [26]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(docs)}")
print(docs[0].page_content)  # Preview the first chunk


Total chunks created: 9
UNDERSTANDING AGENTIC AI
Definition
Agentic AI refers to systems that act autonomously to pursue objectives in environments by perceiving inputs, planning sequences of actions, and executing those actions. Unlike single-turn or purely reactive models, agentic systems maintain state, set sub-goals, chain multiple steps, and can manage decision-making, tool use, and iterative refinement of outputs.
Core Capabilities
Perception & Input Handling
Ingests multi-modal inputs (text, structured data, files, API responses).
Normalizes and encodes inputs into internal representations.
Planning & Reasoning
Breaks a top-level task into sub-tasks.
Generates plans, evaluates alternatives, and sequences actions.
Performs iterative refinement using feedback loops.
Tooling & Action Execution
Calls external tools and APIs (search, code execution, DB queries).
Reads/writes files, uses calculators, or interacts with web services.
Manages side effects with retries, error handling, an

In [27]:
print(f"Total chunks created: {len(text_chunks)}")
print(text_chunks[0].page_content)  # preview the first chunk


Total chunks created: 48
UNDERSTANDING AGENTIC AI


In [23]:
! uv pip install faiss-cpu google-generativeai

Using Python 3.12.2 environment at: e:\projects\llmops_series\.venv
Resolved 33 packages in 1.57s
Prepared 9 packages in 25.11s
Uninstalled 3 packages in 43ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 9 packages in 232ms
 - google-ai-generativelanguage==0.9.0
 + google-ai-generativelanguage==0.6.15
 + google-api-python-client==2.185.0
 + google-auth-httplib2==0.2.0
 + google-generativeai==0.8.5
 - grpcio-status==1.76.0
 + grpcio-status==1.71.2
 + httplib2==0.31.0
 - protobuf==6.33.0
 + protobuf==5.29.5
 + pyparsing==3.2.5
 + uritemplate==4.2.0


In [28]:
import google.generativeai as genai
from langchain.embeddings.base import Embeddings

class GoogleEmbeddings(Embeddings):
    def __init__(self, api_key: str, model: str = 'models/text-embedding-004'):
        self.model = model
        genai.configure(api_key=api_key)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        response = genai.generate_embeddings(model=self.model, text=texts)
        return [embedding['embedding'] for embedding in response.embeddings]

    def embed_query(self, text: str) -> list[float]:
        response = genai.generate_embeddings(model=self.model, text=[text])
        return response.embeddings[0]['embedding']


e:\projects\llmops_series\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
! pip install -U langchain-vectorstores faiss-cpu


ERROR: Could not find a version that satisfies the requirement langchain-vectorstores (from versions: none)
ERROR: No matching distribution found for langchain-vectorstores

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\karthikeya\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [34]:
from langchain_community.vectorstores.faiss import FAISS

In [ ]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [ ]:
vectorstore

In [ ]:
retriever=vectorstore.as_retriever()

In [ ]:
 Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

In [ ]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [ ]:
prompt=ChatPromptTemplate.from_template(template)

In [ ]:
prompt

In [ ]:
from langchain.schema.output_parser import StrOutputParser

In [ ]:
output_parser=StrOutputParser()

In [ ]:
from langchain.chat_models import ChatOpenAI

llm_model=ChatOpenAI(model_name="gpt-4o-mini")

In [ ]:
from langchain.schema.runnable import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [ ]:
rag_chain.invoke("tell me about Agentic AI")

In [ ]:
import structlog